# Dónde termina la preparación de datos y empieza el modelado

Todos los notebooks de este tema transforman datos, y algunas de esas transformaciones se pueden hacer una vez, para todos los que vayan a usar la tabla, mientras que otras hay que aprenderlas de nuevo, a partir de las filas de entrenamiento, para cada modelo. Distinguir unas de otras es lo que mantiene honesta la evaluación de un modelo, y es también lo que decide quién hace cada paso en un proyecto real.

## Dos pipelines

En un proyecto real, los datos suelen pasar por dos pipelines:

- **El pipeline de datos**, a menudo llamado **ETL** (*extract, transform, load*: extraer, transformar, cargar): extrae los datos de donde están (archivos, bases de datos, APIs web), los transforma en una tabla limpia y documentada (tipos, nombres, uniones, reglas de calidad) y los carga donde otros puedan usarlos. Lo construyen los ingenieros de datos, y su tabla sirve para muchos usos: informes, cuadros de mando, varios modelos. La calidad de los datos y el proceso ETL son temas en sí mismos, más allá de este.
- **El pipeline del modelo**: todo lo que hay entre esa tabla y una predicción. En scikit-learn, un `Pipeline` cuyos pasos se ajustan con las filas de entrenamiento y después se aplican sin cambios a cada fila nueva ([Viviendas de California con pipelines](../supervised-learning/basic-workflows/california_with_pipelines.ipynb)).

Muchos pasos podrían ir en cualquiera de los dos. Una pregunta lo decide.

## El criterio

**¿Aprende el paso algo de los datos, o depende de cómo se vayan a dividir los datos para un modelo?**

- Si no, puede formar parte del ETL: da el mismo resultado lo ejecute quien lo ejecute y para el propósito que sea.
- Si sí, pertenece al pipeline del modelo, *después* de la división. Un paso que aprende de los datos debe aprender solo de las filas de entrenamiento, o la información de las filas de validación y de prueba se fuga al modelo, y la evaluación deja de medir lo que debería ([Fuga de datos](../supervised-learning/ml-fundamentals/data_leakage.ipynb)).

| Paso | ¿Aprende de los datos? | Dónde va | Ejemplos |
| --- | --- | --- | --- |
| Extracción y conversión de formato | no | ETL | una hoja de cálculo o las respuestas de una API a una tabla; eliminar una columna que duplica otra |
| Tipos, nombres, unir fuentes | no | ETL | fechas interpretadas, un código leído como texto, nombres de columna descriptivos, una tabla a partir de varios archivos |
| Reglas de calidad de datos: comprobar, informar, apartar | no (reglas fijas) | ETL, y se vuelve a comprobar antes de modelar | marcadores a valores ausentes, valores imposibles, duplicados |
| **Enriquecimiento:** datos de referencia, variables solo a partir de cada fila | no | ETL | la región de una dirección a partir de un mapa fijo; la distancia a un punto fijo; un cociente de dos columnas |
| Imputación, escalado, codificación | sí: medianas, medias, listas de categorías | pipeline del modelo, ajustado con las filas de entrenamiento | una mediana que rellena los huecos; codificación *one-hot* |
| **Variables aprendidas** | sí | pipeline del modelo, ajustado con las filas de entrenamiento | intervalos a partir de cuantiles; clústeres de ubicaciones; el objetivo medio de cada categoría |
| Variables a partir *de los objetivos de otras filas* | sí, del propio objetivo | pipeline del modelo, con el máximo cuidado: el mayor riesgo de fuga | el objetivo medio de una categoría; los precios de los vecinos más cercanos |
| Transformar el objetivo | un logaritmo no, una estandarización sí | pipeline del modelo (`TransformedTargetRegressor`), para que las predicciones vuelvan en las unidades del objetivo | el logaritmo del objetivo |

## Cada paso de este tema

Los pasos de los demás notebooks, según el criterio:

| Paso | ¿Aprende? | Dónde | Notebook |
| --- | --- | --- | --- |
| Eliminar filas sin ninguna medida | no | ETL | [Explorar un dataset](deep-dives/exploring_a_dataset.ipynb) |
| Convertir marcadores y valores imposibles en valores ausentes | no | ETL | [Valores ausentes](missing_values.ipynb), [Valores atípicos](deep-dives/outliers.ipynb) |
| Una columna que dice si un valor se registró (`cabin_known`) | no | cualquiera | [Valores ausentes](missing_values.ipynb) |
| Rellenar huecos con una mediana, el valor más frecuente, vecinos o un modelo | sí | modelo | [Valores ausentes](missing_values.ipynb) |
| Recortar en un límite conocido del dominio (más de 10 baños contados como 10) | no | ETL | [Valores atípicos](deep-dives/outliers.ipynb) |
| Winsorizar en percentiles de los datos | sí | modelo | [Valores atípicos](deep-dives/outliers.ipynb) |
| Codificación ordinal con el orden dado a mano; tratamientos agrupados por significado | no | cualquiera | [Codificar categorías](categorical_encoding.ipynb), [Crear variables](deep-dives/creating_features.ipynb) |
| Codificación *one-hot* (aprende la lista de categorías); agrupar categorías raras; codificación por objetivo | sí | modelo | [Codificar categorías](categorical_encoding.ipynb) |
| Cualquier escalador | sí | modelo | [Escalar variables numéricas](feature_scaling.ipynb) |
| Un logaritmo | no | cualquiera | [Distribuciones asimétricas](deep-dives/skewed_distributions.ipynb) |
| Una transformación de potencia o de cuantiles | sí | modelo | [Distribuciones asimétricas](deep-dives/skewed_distributions.ipynb) |
| Cocientes, tiempo transcurrido, meses cíclicos, productos | no | cualquiera | [Crear variables](deep-dives/creating_features.ipynb) |
| Intervalos con bordes tomados de los datos | sí | modelo | [Crear variables](deep-dives/creating_features.ipynb) |

«Cualquiera» significa que el paso puede hacerse una vez en el ETL, o dentro del pipeline para que el pipeline pueda recibir filas en bruto por sí solo; la última sección, una profundización, vuelve sobre esa elección.

## Una fuga, en una columna de puro ruido

La tabla del censo de [Codificar categorías](categorical_encoding.ipynb), con una columna añadida: un código con 10 000 valores posibles, sorteado al azar para cada persona. No contiene ninguna información sobre los ingresos, por construcción:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

adult = fetch_openml("adult", version=2, as_frame=True, data_home="data").frame
random_generator = np.random.default_rng(0)
X = adult.drop(columns="class").assign(code=random_generator.integers(0, 10_000, len(adult)).astype(str))
y = (adult["class"] == ">50K").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train["code"].value_counts().describe()[["mean", "min", "max"]]

mean     3.990706
min      1.000000
max     14.000000
Name: count, dtype: float64

Unas cuatro filas de entrenamiento por código. Una regresión logística con las demás variables, como en [Codificar categorías](categorical_encoding.ipynb), es la referencia honesta:

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

numerical = X.select_dtypes("number").columns.to_list()
nominal = [column for column in X.select_dtypes("category").columns if column != "education"]


def logistic_with(*extra_columns):
    preprocessing = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), nominal),
        (StandardScaler(), numerical),
        *extra_columns,
    )
    return make_pipeline(preprocessing, LogisticRegression(max_iter=3000))


scores = {"without the code": cross_val_score(logistic_with(), X_train, y_train, cv=5).mean()}
round(scores["without the code"], 3)

np.float64(0.851)

Ahora el código, con codificación por objetivo ([Codificar categorías](categorical_encoding.ipynb)) de tres formas. Primero a mano, con la proporción de ingresos altos entre las filas de entrenamiento de cada código, calculada una vez sobre todas las filas de entrenamiento, antes de la validación cruzada:

In [ ]:
share_by_code = y_train.groupby(X_train["code"]).mean()
X_train_by_hand = X_train.assign(code_encoded=X_train["code"].map(share_by_code))
scores["encoded by hand, before the split"] = cross_val_score(
    logistic_with((StandardScaler(), ["code_encoded"])), X_train_by_hand, y_train, cv=5).mean()

Segundo, con el `TargetEncoder` de scikit-learn, pero ajustado y aplicado a todas las filas de entrenamiento antes de la validación cruzada:

In [ ]:
from sklearn.model_selection import StratifiedKFold

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)  # los pliegues del ajuste cruzado, fijos para que los resultados se repitan
target_encoder = TargetEncoder(cv=folds).fit(X_train[["code"]], y_train)
X_train_fitted_first = X_train.assign(code_encoded=target_encoder.transform(X_train[["code"]]).ravel())
scores["TargetEncoder fitted before the split"] = cross_val_score(
    logistic_with((StandardScaler(), ["code_encoded"])), X_train_fitted_first, y_train, cv=5).mean()

Tercero, `TargetEncoder` dentro del pipeline, para que cada pliegue de la validación cruzada lo ajuste con su propia parte de entrenamiento:

In [ ]:
scores["TargetEncoder inside the pipeline"] = cross_val_score(
    logistic_with((TargetEncoder(cv=folds), ["code"])), X_train, y_train, cv=5).mean()
pd.Series(scores).round(3).rename("cross-validated accuracy")

without the code                         0.851
encoded by hand, before the split        0.892
TargetEncoder fitted before the split    0.893
TargetEncoder inside the pipeline        0.851
Name: cross-validated accuracy, dtype: float64

Una columna de puro ruido ha subido la exactitud cuatro puntos, dos veces: la validación cruzada dice ahora que el modelo es mejor. Dentro del pipeline, el ruido no añade nada, como debe ser.

El motivo: con unas cuatro personas por código, la proporción de ingresos altos de un código es, en gran medida, los propios ingresos de la persona. La columna codificada lleva un trozo de la respuesta de cada fila, y el modelo aprende a leerlo. La validación cruzada no puede ver el problema, porque sus filas de validación también se codificaron con proporciones que incluían sus propios objetivos: la fuga ocurrió antes de la división, así que la división no puede detectarla. Dentro del pipeline, cada pliegue codifica sus filas de validación con proporciones aprendidas solo de otras filas, y con ellas el código es lo que de verdad es, ruido.

Las filas nuevas, el conjunto de prueba, se codifican como cualquier fila futura: con las proporciones aprendidas de las filas de entrenamiento, que nunca incluyen los ingresos de la propia fila nueva:

In [ ]:
leaky_model = logistic_with((StandardScaler(), ["code_encoded"])).fit(X_train_by_hand, y_train)
X_test_encoded = X_test.assign(code_encoded=X_test["code"].map(share_by_code).fillna(y_train.mean()))
baseline_model = logistic_with().fit(X_train, y_train)
pd.Series({
    "the model with the leaked column, as cross-validation predicted": scores["encoded by hand, before the split"],
    "the same model, on the test rows": leaky_model.score(X_test_encoded, y_test),
    "the model without the code, on the test rows": baseline_model.score(X_test, y_test),
}).round(3).rename("accuracy")

the model with the leaked column, as cross-validation predicted    0.892
the same model, on the test rows                                   0.794
the model without the code, on the test rows                       0.852
Name: accuracy, dtype: float64

La fuga prometía 0,89 y dio 0,79, peor que el modelo que nunca vio el código (0,85). El modelo había aprendido a fiarse de una columna que, en filas nuevas, es aleatoria. Esta es la forma general de una fuga: una evaluación mejor que la realidad, que solo se descubre cuando el modelo se encuentra con datos que no ha visto, o nunca. La codificación por objetivo la hace espectacular porque usa el propio objetivo; la imputación y el escalado ajustados con todas las filas también filtran, más discretamente ([Fuga de datos](../supervised-learning/ml-fundamentals/data_leakage.ipynb) mide el mismo error con una selección de variables, y explica por qué la de un escalador pasa desapercibida).

## La ingeniería de variables está a ambos lados *(profundización)*

Una variable definida a partir del conocimiento del dominio y calculada solo a partir de cada fila (una distancia a un punto fijo, la región de una dirección a partir de un mapa fijo, un cociente) puede generarse una vez, en el ETL, para todos los usuarios de la tabla. Una variable cuyos valores dependen de otras filas (un clúster, una media, los vecinos más cercanos) se ajusta, así que pertenece al pipeline del modelo, después de la división. «Ingeniería de variables» nombra ambas cosas, y solo el criterio las separa.

## Armonizar no debe ocultar una diferencia de significado *(profundización)*

Unir fuentes es ETL, y también lo es calcular la misma medida para cada una de ellas. Fusionar los resultados bajo un mismo nombre es otra cosa. Si tres fuentes miden cada una una distancia a un punto de referencia distinto (a una avenida principal distinta en cada ciudad, por ejemplo), la tabla honesta mantiene tres columnas, cada una vacía fuera de su propia fuente: una única columna «distancia al punto de referencia» parecería una armonización y ocultaría que los tres números no significan lo mismo. Si un modelo entrenado con una fuente puede usar el punto de referencia de otra es una pregunta sobre el modelo, no algo que el ETL deba decidir en silencio. Lo mismo vale para los nombres: un ETL honesto da a cada columna un nombre que dice lo que es.

## La línea no siempre es nítida *(profundización)*

Los pasos que no aprenden nada *pueden* ir en el ETL; no tienen por qué. A menudo un equipo calcula igualmente algunos de ellos dentro del pipeline del modelo, para que el pipeline pueda recibir una fila en bruto en producción y calcularlo todo por sí mismo: un modelo desplegado como servicio recibe los mismos campos en bruto que tenían sus datos de entrenamiento, no una tabla preparada por el ETL de otro. El criterio sigue diciendo qué pasos pueden pasar al ETL, y cuáles no deben salir nunca del pipeline. Lo que la producción le pide a un pipeline (las mismas transformaciones en cada fila nueva, y solo las entradas conocidas en el momento de predecir) se explica en [El sobreajuste y la división en entrenamiento y prueba](../supervised-learning/ml-fundamentals/overfitting_and_the_split.ipynb#el-conjunto-de-prueba-representa-la-producción).

Dos ejemplos completos trazan la línea con datos reales: [Diabetes Pima](../supervised-learning/basic-workflows/pima_diabetes.ipynb), donde los ceros imposibles se convierten en valores ausentes antes de la división y se rellenan dentro del pipeline, y [Enfermedad de tiroides, preparación de datos](https://github.com/avidaldo/thyroid/blob/master/01_data_preparation.ipynb), que separa explícitamente su ETL del preprocesamiento del modelo.